# MRR

Compare Monthly Recurring Revenue computed by the Tidemill engine with
ground-truth values derived directly from Stripe subscription objects.

## How Stripe stores recurring prices

Every Stripe **Subscription** contains one or more **SubscriptionItems**,
each referencing a **Price** object.  The Price defines:

| Field                       | Meaning                                             |
|-----------------------------|-----------------------------------------------------|
| `unit_amount`               | Price in smallest currency unit (e.g. 2100 = $21)   |
| `recurring.interval`        | Billing cycle: `day`, `week`, `month`, or `year`    |
| `recurring.interval_count`  | Intervals per cycle (e.g. 3 → quarterly if monthly) |

A SubscriptionItem also carries a `quantity` (default 1), so the total
charge per cycle is `unit_amount × quantity`.

## How MRR is normalised

MRR converts every billing interval to a **monthly** rate:

| Interval | Formula                              |
|----------|--------------------------------------|
| month    | `amount / interval_count`            |
| year     | `amount / (12 × interval_count)`     |
| week     | `amount × 52 / (12 × interval_count)`|
| day      | `amount × 365 / (12 × interval_count)`|

Only subscriptions with `status == "active"` contribute to current MRR.
All arithmetic uses integer cents to avoid floating-point drift.

In [1]:
import os

import stripe

from tidemill import reports
from tidemill.reports.stripecheck import StripeData, TidemillClient
from tidemill.reports.stripecheck import compare as cmp

stripe.api_key = os.environ["STRIPE_API_KEY"]
reports.setup()

START, END = "2025-09-01", "2026-04-30"
tm = TidemillClient()
sd = StripeData()
print(f"Stripe subscriptions: {sd.summary()}")

Stripe subscriptions: 38 total, 26 active, 11 canceled, 1 incomplete_expired


## 1. MRR comparison — Tidemill vs Stripe

`stripecheck.compare.mrr()` fetches Tidemill's MRR via the REST API
and independently sums `mrr_cents` over active Stripe subscriptions.
Any difference indicates a divergence in event processing.

In [2]:
reports.mrr.style_stripe_comparison(reports.mrr.stripe_comparison(tm, sd, at="2026-03-01"))

MRR (Tidemill),MRR (Stripe),Difference,Match,ARR
"$1,009.33","$1,009.33",$0.00,True,"$12,111.96"


## 2. Per-subscription MRR breakdown

Drill into every Stripe subscription to see its individual MRR
contribution.  `stripecheck.stripe_metrics.subscription_mrr()` reads
`items.data[*].price` on each subscription and applies the normalisation
table from above.  Useful for identifying *which* subscription diverges
when the totals don't match.

In [3]:
cmp.per_subscription_mrr(sd)

,id,customer,status,mrr_cents,currency,mrr
0,sub_1TLTmNCMLOTbZAd7QIgUVBba,cus_UK82U19J0AxPeb,active,20750,usd,$207.50
1,sub_1TLTmaCMLOTbZAd7uAxWFKqh,cus_UK82F2IfibEKGK,canceled,7900,usd,$79.00
2,sub_1TLTm5CMLOTbZAd787F67ssP,cus_UK82dhnKW6sAca,active,7900,usd,$79.00
3,sub_1TLTmACMLOTbZAd7T5dHdvuO,cus_UK82RyH3jFhZCP,active,7900,usd,$79.00
4,sub_1TLTmfCMLOTbZAd798D9LJgC,cus_UK82qllkdii0tg,active,7900,usd,$79.00
5,sub_1TLTmkCMLOTbZAd7nbo5Vpvd,cus_UK82Vo4XKxFerh,active,7900,usd,$79.00
6,sub_1TLTmHCMLOTbZAd7f2xktOz8,cus_UK82Y9MKs65Xrc,active,6583,usd,$65.83
7,sub_1TLTmvCMLOTbZAd7Fwe23xy7,cus_UK82AJEh7Ru5Td,incomplete_expired,2100,usd,$21.00
8,sub_1TLTnCCMLOTbZAd7XYmuYyLe,cus_UK835QXbL8AMHU,active,2100,usd,$21.00
9,sub_1TLTn8CMLOTbZAd7mDK6NGQC,cus_UK83reKGoJUjfj,active,2100,usd,$21.00


## 3. MRR Breakdown (movements)

Tidemill classifies every MRR change into a **movement type**:

| Movement      | Meaning                                              |
|---------------|------------------------------------------------------|
| **new**       | First subscription for a customer                    |
| **expansion** | Upgrade or additional subscription                   |
| **contraction** | Downgrade (lower plan/fewer seats)                 |
| **churn**     | Cancellation — customer lost entirely                |
| **reactivation** | Previously churned customer returns               |

The breakdown endpoint sums these over the full period.

In [4]:
reports.mrr.plot_breakdown(reports.mrr.breakdown(tm, START, END))

## 4. Monthly MRR with movement split

Query MRR breakdown per month to show how new, expansion, contraction, and churn compose each month's MRR change.

In [5]:
wf = reports.mrr.waterfall(tm, START, END)
reports.mrr.style_waterfall(wf)

,starting_mrr,new,expansion,reactivation,contraction,churn,net_change,ending_mrr
month,,,,,,,,
2025-09,$0.00,$715.33,$0.00,$0.00,$0.00,$-42.00,$673.33,$673.33
2025-10,$673.33,$231.00,$116.00,$0.00,$-58.00,$-42.00,$247.00,$920.33
2025-11,$920.33,$0.00,$0.00,$0.00,$0.00,$-121.00,$-121.00,$799.33
2025-12,$799.33,$168.00,$0.00,$0.00,$0.00,$-42.00,$126.00,$925.33
2026-01,$925.33,$105.00,$0.00,$0.00,$0.00,$-21.00,$84.00,"$1,009.33"
2026-02,"$1,009.33",$0.00,$0.00,$0.00,$0.00,$-42.00,$-42.00,$967.33
2026-03,$967.33,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$967.33
2026-04,$967.33,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$967.33


In [6]:
reports.mrr.plot_waterfall(wf)

In [7]:
reports.mrr.plot_trend(reports.mrr.trend(tm, START, END))

## 5. MRR by subscription status (Stripe)

Stripe subscriptions move through lifecycle states:

| Status               | Pays? | Notes                                       |
|----------------------|-------|---------------------------------------------|
| `trialing`           | No    | Free trial period, no charge yet            |
| `active`             | Yes   | Current, paying — contributes to MRR        |
| `past_due`           | Retry | Payment failed, Stripe retrying             |
| `canceled`           | No    | Terminated — `canceled_at` is set           |
| `unpaid`             | No    | Retries exhausted, still open               |
| `incomplete`         | No    | First payment pending                       |
| `incomplete_expired` | No    | First payment never completed               |

Only `active` subscriptions are counted toward MRR.

In [8]:
status_df = reports.mrr.stripe_status_breakdown(sd)
reports.mrr.style_stripe_status_breakdown(status_df)

status,count,mrr
active,26,"$1,009.33"
canceled,11,$289.00
incomplete_expired,1,$21.00


In [9]:
reports.mrr.plot_stripe_status_breakdown(status_df)